In [1]:
!nvidia-smi
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Sat Apr  4 20:37:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import time
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from sklearn.model_selection import train_test_split

In [3]:
# Importing Datasets from Kaggle

from google.colab import userdata
!export KAGGLE_API_TOKEN={userdata.get('KAGGLE_API_TOKEN')}

import kagglehub

# Download latest version of datasets
ds2 = kagglehub.dataset_download("jeftaadriel/oia-odir-dataset")

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `export KAGGLE_API_TOKEN={userdata.get('KAGGLE_API_TOKEN')}'
Using Colab cache for faster access to the 'oia-odir-dataset' dataset.


In [4]:
print(ds2)

/kaggle/input/oia-odir-dataset


In [5]:
# Convert files to csv format
off_site_file = pd.read_excel(f"{ds2}/Off-site Test Set/Annotation/off-site test annotation (English).xlsx")
off_site_file.to_csv("off_site_test_annotation.csv", index=None, header=True)

on_site_file = pd.read_excel(f"{ds2}/On-site Test Set/Annotation/on-site test annotation (English).xlsx")
on_site_file.to_csv("on_site_test_annotation.csv", index=None, header=True)

training_file = pd.read_excel(f"{ds2}/Training Set/Annotation/training annotation (English).xlsx")
training_file.to_csv("training_annotation.csv", index=None, header=True)

# Base paths for images
base_path_off_site = ds2 + '/Off-site Test Set/Images/'
base_path_on_site = ds2 + '/On-site Test Set/Images/'
base_path_training = ds2 + '/Training Set/Images/'

# Add image paths to a DataFrame
def process_fundus_image_paths(df, base_path, left_col='Left-Fundus', right_col='Right-Fundus'):
    def add_image_path(filename, current_base_path):
        if pd.isna(filename):
            return filename
        return os.path.join(current_base_path, filename)

    df[left_col] = df[left_col].apply(lambda x: add_image_path(x, base_path))
    df[right_col] = df[right_col].apply(lambda x: add_image_path(x, base_path))
    return df

# Apply to each DataFrame
off_site_file = process_fundus_image_paths(off_site_file, base_path_off_site)
on_site_file = process_fundus_image_paths(on_site_file, base_path_on_site)
training_file = process_fundus_image_paths(training_file, base_path_training)

# Combine all DataFrames into one
df_ds2 = pd.concat([off_site_file, on_site_file, training_file], ignore_index=True)

# Remove specified columns
columns_to_remove = ['ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
df_ds2 = df_ds2.drop(columns=columns_to_remove)

# Convert Keywords into diagnosis for each eye
disease_mapping = {
    'normal': 'NORMAL',
    'diabetic retinopathy': 'DIABETIC_RETINOPATHY',
    'glaucoma': 'GLAUCOMA',
    'cataract': 'CATARACT',
    'macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'age-related macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'hypertensive retinopathy': 'HYPERTENSION',
    'myopia': 'MYOPIA',
}

df_left = df_ds2[['Left-Fundus', 'Left-Diagnostic Keywords']].copy()
df_right = df_ds2[['Right-Fundus', 'Right-Diagnostic Keywords']].copy()

df_left.columns = ['image_path', 'eye_disease']
df_right.columns = ['image_path', 'eye_disease']

# Get unique disease codes for column names
disease_codes = list(set(disease_mapping.values()))
disease_codes.append('OTHER_DISEASES')

def parse_diagnosis(df):
    # Initialize all disease columns with 0
    for disease in disease_codes:
        df[disease] = 0

    # Parse each row's diagnostic keywords
    for idx, keywords in df['eye_disease'].items():
        if pd.isna(keywords):
            continue

        # Convert to lowercase for matching
        keywords_lower = str(keywords).lower()

        disease_found = False

        # Check for each disease keyword in the mapping
        for keyword, disease_code in disease_mapping.items():
            if keyword in keywords_lower:
                df.at[idx, disease_code] = 1
                disease_found = True

        if not disease_found:
            df.at[idx, 'OTHER_DISEASES'] = 1

    df = df.drop('eye_disease', axis=1)

    return df

# Apply to both dataframes
df_left = parse_diagnosis(df_left)
df_right = parse_diagnosis(df_right)

# Combine to final dataset
df_ds2 = pd.concat([df_left, df_right], ignore_index=True)

print(df_left)
print(df_right)

                                             image_path  CATARACT  NORMAL  \
0     /kaggle/input/oia-odir-dataset/Off-site Test S...         0       0   
1     /kaggle/input/oia-odir-dataset/Off-site Test S...         0       0   
2     /kaggle/input/oia-odir-dataset/Off-site Test S...         0       0   
3     /kaggle/input/oia-odir-dataset/Off-site Test S...         0       0   
4     /kaggle/input/oia-odir-dataset/Off-site Test S...         0       0   
...                                                 ...       ...     ...   
4995  /kaggle/input/oia-odir-dataset/Training Set/Im...         0       0   
4996  /kaggle/input/oia-odir-dataset/Training Set/Im...         0       0   
4997  /kaggle/input/oia-odir-dataset/Training Set/Im...         0       0   
4998  /kaggle/input/oia-odir-dataset/Training Set/Im...         0       0   
4999  /kaggle/input/oia-odir-dataset/Training Set/Im...         0       0   

      HYPERTENSION  AGE_RELATED_MACULAR_DEGENERATION  DIABETIC_RETINOPATHY 

In [32]:
class EyeDiseaseDataset(Dataset):
    def __init__(self, dataframe, label_cols, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_cols = label_cols
        self.labels = self.dataframe[self.label_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, "image_path"]
        image = Image.open(img_path).convert("RGB")

        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [33]:
def crop_black_borders(image):
    # Convert PIL Image to NumPy array
    img_np = np.array(image)

    # Threshold for black pixels
    # Pixels with R, G, B values all below this threshold will be considered black
    BLACK_THRESHOLD = 10

    non_black_pixels = np.any(img_np > BLACK_THRESHOLD, axis=2)

    # Find rows and columns that contain non-black pixels
    rows = np.any(non_black_pixels, axis=1)
    cols = np.any(non_black_pixels, axis=0)

    # Get the min/max indices for cropping
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    # Crop the NumPy array
    cropped_img_np = img_np[rmin:rmax+1, cmin:cmax+1]

    # Convert back to PIL Image
    return Image.fromarray(cropped_img_np)

In [35]:
BATCH_SIZE = 128
NUM_EPOCHS = 5
LR = 1e-4
NUM_WORKERS = 2
IMAGE_SIZE=224

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "best_efficientnet_b0.pth"

SEED = 68

In [36]:
import random
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
set_seed(SEED)

In [37]:
disease_cols = [
    "HYPERTENSION",
    "CATARACT",
    "AGE_RELATED_MACULAR_DEGENERATION",
    "GLAUCOMA",
    "MYOPIA",
    "DIABETIC_RETINOPATHY",
    "NORMAL",
    "OTHER_DISEASES",
]

# Keep only rows with exactly one positive label
df = df_ds2.copy()
df["num_labels"] = df[disease_cols].sum(axis=1)
df_single = df[df["num_labels"] == 1].copy()

# Convert one-hot columns into a single target label
df_single["target_name"] = df_single[disease_cols].idxmax(axis=1)

class_names = disease_cols
label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

df_single["target"] = df_single["target_name"].map(label_to_idx)

# Keep only rows whose image path exists
df_single = df_single[df_single["image_path"].apply(lambda x: isinstance(x, str))].copy()
df_single = df_single.reset_index(drop=True)

print("Original rows:", len(df))
print("Single-label rows kept:", len(df_single))
print(df_single[["image_path", "target_name", "target"]].head())

Original rows: 10000
Single-label rows kept: 9895
                                          image_path     target_name  target
0  /kaggle/input/oia-odir-dataset/Off-site Test S...    HYPERTENSION       0
1  /kaggle/input/oia-odir-dataset/Off-site Test S...  OTHER_DISEASES       7
2  /kaggle/input/oia-odir-dataset/Off-site Test S...  OTHER_DISEASES       7
3  /kaggle/input/oia-odir-dataset/Off-site Test S...  OTHER_DISEASES       7
4  /kaggle/input/oia-odir-dataset/Off-site Test S...  OTHER_DISEASES       7


In [38]:
train_df, test_df = train_test_split(
    df_single,
    test_size=0.2,
    stratify=df_single["target"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Val size:", len(test_df))

Train size: 7916
Val size: 1979


In [39]:
class ODIRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        target = int(row["target"])

        if self.transform is not None:
            image = self.transform(image)

        return image, target

In [40]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [41]:
train_dataset = ODIRDataset(train_df, transform=train_transform)
test_dataset = ODIRDataset(test_df, transform=test_transform)

In [42]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [43]:
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [45]:
num_classes = len(df_ds2.columns) - 1

model = models.shufflenet_v2_x1_0(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model = model.to(device)

print("Output classes:", model.fc.out_features)

Output classes: 8


In [46]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [47]:
from tqdm.auto import tqdm

In [48]:
# training function
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_num=None, num_epochs=None):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_num}/{num_epochs} [Train]" if epoch_num is not None else "Training",
        leave=False
    )

    for batch_idx, (images, labels) in enumerate(progress_bar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        avg_loss = running_loss / (batch_idx * images.size(0))

        progress_bar.set_postfix(
            batch_loss=f"{loss.item():.4f}",
            avg_loss=f"{avg_loss:.4f}"
        )

    return running_loss / len(loader.dataset)

In [49]:
# evaluation function
def evaluate(model, loader, criterion, device, epoch_num=None, num_epochs=None):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_labels = []

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_num}/{num_epochs} [Test]" if epoch_num is not None else "Evaluating",
        leave=False
    )

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(progress_bar, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)

            running_loss += loss.item() * images.size(0)
            avg_loss = running_loss / (batch_idx * images.size(0))

            progress_bar.set_postfix(
                batch_loss=f"{loss.item():.4f}",
                avg_loss=f"{avg_loss:.4f}"
            )

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return running_loss / len(loader.dataset), all_probs, all_labels

In [50]:
sample_img, sample_label = train_dataset[0]
print("single label shape:", sample_label.shape)
print("single label:", sample_label)

AttributeError: 'int' object has no attribute 'shape'

In [30]:
# training loop
num_epochs = 5
best_test_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device,
        epoch_num=epoch + 1, num_epochs=num_epochs
    )

    test_loss, test_probs, test_labels = evaluate(
        model, test_loader, criterion, device,
        epoch_num=epoch + 1, num_epochs=num_epochs
    )

    scheduler.step()

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_shufflenet_eye_disease.pth")

    epoch_time = time.time() - start_time

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    print(f"Time:       {epoch_time:.1f}s")
    print("-" * 30)

model.load_state_dict(best_model_wts)

Epoch 1/5 [Train]:   0%|          | 0/62 [00:00<?, ?it/s]

ValueError: Target size (torch.Size([128])) must be the same as input size (torch.Size([128, 8]))